In [2]:
# Cell 1: Import mô-đun và Tải dữ liệu thô
import sys
import os
import pandas as pd
import numpy as np

sys.path.append(os.path.abspath(".."))
from utils.data_fetcher import StockDataFetcher
from utils.data_preprocessor import DataPreprocessor

# 1. Khởi tạo Fetcher & Preprocessor
fetcher = StockDataFetcher()
preprocessor = DataPreprocessor()

# 2. Đọc dữ liệu thô GAS đã tải từ Notebook 02
df_raw = fetcher.load_from_sqlite("MBB")
print("=== DỮ LIỆU BAN ĐẦU ===")
print(df_raw.info())
print(f"Số lượng missing values ban đầu:\n{df_raw.isnull().sum()}")

# Cell 2: Làm sạch & Điền khuyết an toàn
df_clean = preprocessor.clean_time_series(df_raw, freq="B")

# Cell 3: Xử lý Outliers bằng IQR Clipping
cols_to_check = ["Open", "High", "Low", "Close", "Volume"]
df_no_outliers = preprocessor.handle_outliers_iqr(df_clean, columns=cols_to_check, factor=1.5)

# Cell 4: Thử nghiệm Resample sang khung Tuần (Weekly)
df_weekly = preprocessor.resample_ohlcv(df_no_outliers, rule="W")
print("\n=== DỮ LIỆU KHUNG TUẦN (WEEKLY OHLCV) ===")
print(df_weekly.head())

# Cell 5: Thực hiện Scaling (Robust Scaler)
df_scaled = preprocessor.fit_transform_scale(df_no_outliers, columns=["Close", "Volume"], method="robust")
print("\n=== DỮ LIỆU SAU KHI ROBUST SCALING ===")
print(df_scaled[["Close", "Volume"]].head())

# Cell 6: Lưu dữ liệu đã tiền xử lý vào data/processed/
os.makedirs("../data/processed", exist_ok=True)
# Sửa tên file từ AAPL_cleaned.parquet thành GAS_cleaned.parquet
processed_path = "../data/processed/MBB_cleaned.parquet" 
df_no_outliers.to_parquet(processed_path, engine="pyarrow")
print(f"\n[Thành công] Đã lưu dữ liệu sạch vào: {processed_path}")

2026-07-31 15:26:27,797 [INFO] Đã làm sạch dữ liệu. Kích thước sau xử lý: (1566, 5)
2026-07-31 15:26:27,808 [INFO] Cột 'Open': Phát hiện & Clip 77 giá trị ngoại lệ.
2026-07-31 15:26:27,817 [INFO] Cột 'High': Phát hiện & Clip 79 giá trị ngoại lệ.
2026-07-31 15:26:27,824 [INFO] Cột 'Low': Phát hiện & Clip 76 giá trị ngoại lệ.
2026-07-31 15:26:27,832 [INFO] Cột 'Close': Phát hiện & Clip 78 giá trị ngoại lệ.
2026-07-31 15:26:27,837 [INFO] Cột 'Volume': Phát hiện & Clip 74 giá trị ngoại lệ.
2026-07-31 15:26:27,852 [INFO] Đã Resample dữ liệu sang khung 'W'. Kích thước mới: (314, 5)
2026-07-31 15:26:27,866 [INFO] Đã Scale các cột ['Close', 'Volume'] bằng phương pháp 'robust'.


=== DỮ LIỆU BAN ĐẦU ===
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1565 entries, 2020-01-01 to 2025-12-31
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Open    1565 non-null   float64
 1   High    1565 non-null   float64
 2   Low     1565 non-null   float64
 3   Close   1565 non-null   float64
 4   Volume  1565 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 73.4 KB
None
Số lượng missing values ban đầu:
Open      0
High      0
Low       0
Close     0
Volume    0
dtype: int64

=== DỮ LIỆU KHUNG TUẦN (WEEKLY OHLCV) ===
                   Open         High          Low        Close      Volume
Date                                                                      
2020-01-05  5797.140625  5992.235754  5797.140072  5922.558594  20571778.0
2020-01-12  5894.688020  6034.042398  5769.269043  5964.365234  69434718.0
2020-01-19  5978.300929  6201.267495  5922.559162  6131.590332  78221389.0
2020-01-26  6